<a href="https://colab.research.google.com/github/Farzanehnaderi/Tehran_ENUI_Urban_Monitoring_GEE/blob/main/ENUI_Tehran_province_2020.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
import geemap
import numpy as np
import pandas as pd

ee.Authenticate()
ee.Initialize(
    project="bold-physics-478216-b4"
)

print("Google Earth Engine initialized successfully.")

In [ ]:
tehran = ee.FeatureCollection(
    "projects/bold-physics-478216-b4/assets/Teharn_province"
)

print("Number of features:", tehran.size().getInfo())

In [ ]:
roi = tehran.geometry()

print("ROI loaded successfully.")

In [ ]:
Map = geemap.Map(basemap="SATELLITE")
Map.centerObject(roi, 8)
Map.addLayer(roi, {"color": "red"}, "Tehran Province")
Map.add_layer_control()
Map

In [ ]:
landsat = ee.ImageCollection(
    "LANDSAT/LC08/C02/T1_L2"
)

landsat_tehran = (
    landsat
    .filterBounds(roi)
)

landsat_2020 = (
    landsat_tehran
    .filterDate(
        "2020-01-01",
        "2021-01-01"
    )
)

print(
    "Number of Landsat images:",
    landsat_2020.size().getInfo()
)

In [ ]:
def mask_landsat_clouds(image):

    qa = image.select("QA_PIXEL")

    cloud = 1 << 3
    cloud_shadow = 1 << 4
    cirrus = 1 << 2
    fill = 1 << 0

    mask = (
        qa.bitwiseAnd(fill).eq(0)
        .And(qa.bitwiseAnd(cloud).eq(0))
        .And(qa.bitwiseAnd(cloud_shadow).eq(0))
        .And(qa.bitwiseAnd(cirrus).eq(0))
    )

    return image.updateMask(mask)

In [ ]:
landsat_clean = (
    landsat_tehran
    .filterDate(
        "2020-01-01",
        "2021-01-01"
    )
    .filter(
        ee.Filter.lt(
            "CLOUD_COVER",
            50
        )
    )
    .map(
        mask_landsat_clouds
    )
)

print(
    "Number of clean Landsat images:",
    landsat_clean.size().getInfo()
)

In [ ]:
image_clean_2020 = (
    landsat_clean
    .median()
    .clip(roi)
)

print(
    "Bands:",
    image_clean_2020.bandNames().getInfo()
)

In [ ]:
image_clean_rgb = (
    image_clean_2020
    .select(
        [
            "SR_B4",
            "SR_B3",
            "SR_B2"
        ]
    )
    .multiply(0.0000275)
    .add(-0.2)
)
image_clean_2020_sr = (
    image_clean_2020
    .select([
        "SR_B1",
        "SR_B2",
        "SR_B3",
        "SR_B4",
        "SR_B5",
        "SR_B6",
        "SR_B7"
    ])
    .multiply(0.0000275)
    .add(-0.2)
)

In [ ]:
Map_RGB = geemap.Map(basemap="SATELLITE")

Map_RGB.centerObject(roi, 8)

Map_RGB.addLayer(
    image_clean_rgb,
    {
        "min": 0,
        "max": 0.3,
        "gamma": 1.3
    },
    "Landsat 2020 RGB"
)

Map_RGB.addLayer(
    roi,
    {"color": "red"},
    "Tehran Province"
)

Map_RGB.add_layer_control()

Map_RGB

In [ ]:
image_clean_2020 = (
    landsat_clean
    .median()
    .clip(roi)
)

Map_RGB_clean = geemap.Map()

Map_RGB_clean.centerObject(roi, 8)

Map_RGB_clean.addLayer(
    image_clean_2020,
    {
        "bands": ["SR_B4", "SR_B3", "SR_B2"],
        "min": 0,
        "max": 3000
    },
    "Landsat 2020 - Cloud Masked"
)

Map_RGB_clean.addLayer(
    tehran.style(
        color="red",
        fillColor="00000000",
        width=2
    ),
    {},
    "Tehran Province"
)

Map_RGB_clean.add_layer_control()

Map_RGB_clean

In [ ]:
print("Original 2020 images:")
print(landsat_2020.size().getInfo())

print("\nCloud-masked images:")
print(landsat_clean.size().getInfo())

In [ ]:
test_image = landsat_clean.first()

print("Bands:")
print(test_image.bandNames().getInfo())

In [ ]:
def mask_landsat_clouds(image):

    qa = image.select("QA_PIXEL")

    # Bits:
    # 1 = Dilated Cloud
    # 2 = Cirrus
    # 3 = Cloud
    # 4 = Cloud Shadow

    mask = (
        qa.bitwiseAnd(1 << 1).eq(0)
        .And(qa.bitwiseAnd(1 << 2).eq(0))
        .And(qa.bitwiseAnd(1 << 3).eq(0))
        .And(qa.bitwiseAnd(1 << 4).eq(0))
    )

    return image.updateMask(mask)

In [ ]:
landsat_clean = (
    landsat
    .filterBounds(roi)
    .filterDate(
        "2020-01-01",
        "2020-12-31"
    )
    .filter(
        ee.Filter.lt(
            "CLOUD_COVER",
            50
        )
    )
    .map(mask_landsat_clouds)
)

print(
    "Clean Landsat images:",
    landsat_clean.size().getInfo()
)

In [ ]:
image_clean_2020 = (
    landsat_clean
    .median()
    .clip(roi)
)
image_clean_rgb = (
    image_clean_2020
    .select([
        "SR_B4",
        "SR_B3",
        "SR_B2"
    ])
    .multiply(0.0000275)
    .add(-0.2)
)

Map_RGB_clean = geemap.Map()

Map_RGB_clean.centerObject(roi, 8)

Map_RGB_clean.addLayer(
    image_clean_rgb,
    {
        "min": 0,
        "max": 0.3,
        "gamma": 1.3
    },
    "Landsat 2020 RGB - Cloud Masked"
)

Map_RGB_clean.addLayer(
    tehran.style(
        color="red",
        fillColor="00000000",
        width=2
    ),
    {},
    "Tehran Province Boundary"
)

Map_RGB_clean.add_layer_control()

Map_RGB_clean

In [ ]:
print(
    image_clean_2020
    .select(["SR_B4", "SR_B3", "SR_B2"])
    .reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=roi,
        scale=30,
        maxPixels=1e9
    )
    .getInfo()
)

In [ ]:
ndvi = (
    image_clean_2020_sr
    .normalizedDifference(
        ["SR_B5", "SR_B4"]
    )
    .rename("NDVI")
)

In [ ]:
Map_NDVI = geemap.Map()

Map_NDVI.centerObject(roi, 8)

Map_NDVI.addLayer(
    ndvi,
    {
        "min": -0.35,
        "max": 0.55,
        "palette": [
            "blue",
            "brown",
            "yellow",
            "lightgreen",
            "darkgreen"
        ]
    },
    "NDVI 2020"
)


Map_NDVI.addLayer(
    tehran.style(
        color="red",
        fillColor="00000000",
        width=2
    ),
    {},
    "Tehran Boundary"
)


Map_NDVI.add_legend(
    title="NDVI",
    legend_dict={
        "Water / No vegetation": "blue",
        "Low vegetation": "brown",
        "Moderate vegetation": "yellow",
        "High vegetation": "lightgreen",
        "Dense vegetation": "darkgreen"
    }
)

Map_NDVI.add_layer_control()

Map_NDVI

In [ ]:
ndvi_stats = ndvi.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=roi,
    scale=30,
    maxPixels=1e9
)

print(ndvi_stats.getInfo())

In [ ]:
ndvi_mean = ndvi.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=roi,
    scale=30,
    maxPixels=1e9
)

print(ndvi_mean.getInfo())

In [ ]:
ndbi_2020 = (
    image_clean_2020_sr
    .normalizedDifference(
        ["SR_B6", "SR_B5"]
    )
    .rename("NDBI")
)

In [ ]:
Map_NDBI = geemap.Map()

Map_NDBI.centerObject(roi, 8)


Map_NDBI.addLayer(
    ndbi_2020,
    {
        "min": -0.5,
        "max": 0.5,
        "palette": [
            "blue",
            "white",
            "red"
        ]
    },
    "NDBI 2020"
)


Map_NDBI.addLayer(
    tehran.style(
        color="red",
        fillColor="00000000",
        width=2
    ),
    {},
    "Boundary"
)


Map_NDBI.add_legend(
    title="NDBI",
    legend_dict={
        "Low Built-up": "blue",
        "Neutral": "white",
        "High Built-up": "red"
    }
)


Map_NDBI.add_layer_control()

Map_NDBI

In [ ]:
ndbi_stats = ndbi_2020.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=roi,
    scale=30,
    maxPixels=1e9
)

print(ndbi_stats.getInfo())

In [ ]:
ndbi_mean = ndbi_2020.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=roi,
    scale=30,
    maxPixels=1e9
)

print(ndbi_mean.getInfo())

In [ ]:
ndwi_2020 = (
    image_clean_2020_sr
    .normalizedDifference(
        ["SR_B3", "SR_B5"]
    )
    .rename("NDWI")
)

In [ ]:
Map_NDWI = geemap.Map()

Map_NDWI.centerObject(roi, 8)


Map_NDWI.addLayer(
    ndwi_2020,
    {
        "min": -0.5,
        "max": 0.5,
        "palette": [
            "brown",
            "white",
            "blue"
        ]
    },
    "NDWI 2020"
)


Map_NDWI.addLayer(
    tehran.style(
        color="red",
        fillColor="00000000",
        width=2
    ),
    {},
    "Tehran Boundary"
)


Map_NDWI.add_legend(
    title="NDWI",
    legend_dict={
        "Low Water / Dry Surface": "brown",
        "Moderate": "white",
        "High Water Content": "blue"
    }
)


Map_NDWI.add_layer_control()

Map_NDWI

In [ ]:
ndwi_stats = ndwi_2020.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=roi,
    scale=30,
    maxPixels=1e9
)

print(ndwi_stats.getInfo())

In [ ]:
ndwi_mean = ndwi_2020.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=roi,
    scale=30,
    maxPixels=1e9
)

print(ndwi_mean.getInfo())

In [ ]:
viirs_2020 = (
    ee.ImageCollection(
        "NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG"
    )
    .filterDate(
        "2020-01-01",
        "2021-01-01"
    )
    .select("avg_rad")
    .mean()
    .clip(roi)
)

print(
    "VIIRS band:",
    viirs.bandNames().getInfo()
)

In [ ]:
Map_VIIRS = geemap.Map()

Map_VIIRS.centerObject(
    roi,
    8
)

Map_VIIRS.addLayer(
    viirs_2020,
    {
        "min": 0,
        "max": 60,
        "palette": [
            "black",
            "purple",
            "blue",
            "cyan",
            "yellow",
            "orange",
            "red"
        ]
    },
    "VIIRS Night Light 2020"
)


Map_VIIRS.addLayer(
    tehran.style(
        color="white",
        fillColor="00000000",
        width=2
    ),
    {},
    "Tehran Boundary"
)


# Legend
Map_VIIRS.add_legend(
    title="VIIRS Night Light",
    legend_dict={
        "Very Low Light": "black",
        "Low Light": "purple",
        "Moderate Light": "blue",
        "Medium Light": "cyan",
        "High Light": "yellow",
        "Very High Light": "orange",
        "Extreme Light": "red"
    }
)


Map_VIIRS.add_layer_control()

Map_VIIRS

In [ ]:
viirs_stats = viirs.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=roi,
    scale=500,
    maxPixels=1e9
)

print("===== VIIRS Statistics =====")
print(viirs_stats.getInfo())

In [ ]:
viirs_mean = viirs_2020.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=roi,
    scale=500,
    maxPixels=1e9
)

print(viirs_mean.getInfo())

In [ ]:
viirs_30m = (
    viirs
    .resample("bilinear")
)

print(
    "VIIRS 30m prepared successfully."
)

In [ ]:
ndbi = (
    image_clean_2020_sr
    .normalizedDifference(
        ["SR_B6", "SR_B5"]
    )
    .rename("NDBI")
)

ndbi_binary = (
    ndbi
    .gt(0)
    .rename("NDBIB")
)

print("NDBI and NDBIB created successfully.")

In [ ]:
ENUI = (
    viirs_30m
    .multiply(ndvi)
    .multiply(ndwi_2020)
    .multiply(ndbi_binary)
    .rename("ENUI")
)

print(
    "ENUI created successfully."
)

In [ ]:
enui_stats = ENUI.reduceRegion(
    reducer=ee.Reducer.minMax()
        .combine(
            reducer2=ee.Reducer.mean(),
            sharedInputs=True
        ),
    geometry=roi,
    scale=30,
    maxPixels=1e9
)

print("===== ENUI Statistics =====")
print(enui_stats.getInfo())

In [ ]:
viirs_check = viirs.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=roi,
    scale=500,
    maxPixels=1e9
)

print("===== VIIRS Check =====")
print(viirs_check.getInfo())

In [ ]:
print("===== ENUI Components Check =====")
ndvi_factor = (
    ee.Image(1)
    .subtract(ndvi)
    .rename("NDVI_factor")
)
ndvi_factor_stats = ndvi_factor.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=roi,
    scale=30,
    maxPixels=1e9
)
ndwi = (
    image_clean_2020_sr
    .normalizedDifference(
        ["SR_B3", "SR_B5"]
    )
    .rename("NDWI")
)

ndwi_binary = (
    ndwi
    .gt(-0.1)
    .rename("NDWIB")
)

ndwi_factor = (
    ee.Image(1)
    .subtract(ndwi_binary)
    .rename("NDWI_factor")
)
ndwi_factor_stats = ndwi_factor.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=roi,
    scale=30,
    maxPixels=1e9
)

ndbi_binary_stats = ndbi_binary.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=roi,
    scale=30,
    maxPixels=1e9
)

print("NDVI factor:", ndvi_factor_stats.getInfo())
print("NDWI factor:", ndwi_factor_stats.getInfo())
print("NDBI binary:", ndbi_binary_stats.getInfo())

In [ ]:
ENUI = (
    viirs_30m
    .multiply(ndvi_factor)
    .multiply(ndwi_factor)
    .multiply(ndbi_binary)
    .rename("ENUI")
)

print("ENUI created successfully.")

In [ ]:
enui_stats = ENUI.reduceRegion(
    reducer=ee.Reducer.minMax()
        .combine(
            reducer2=ee.Reducer.mean(),
            sharedInputs=True
        ),
    geometry=roi,
    scale=30,
    maxPixels=1e9
)

print("===== ENUI Statistics =====")
print(enui_stats.getInfo())

In [ ]:
Map_ENUI = geemap.Map(basemap="SATELLITE")

Map_ENUI.centerObject(roi, 8)

vis_params = {
    "min": 0,
    "max": 2.3,
    "palette": [
        "black",
        "blue",
        "cyan",
        "yellow",
        "red"
    ]
}

Map_ENUI.addLayer(
    ENUI,
    vis_params,
    "ENUI 2020"
)

Map_ENUI.addLayer(
    roi,
    {"color": "white"},
    "Tehran Province"
)

# Add legend
Map_ENUI.add_colorbar(
    vis_params,
    label="ENUI",
    layer_name="ENUI 2020",
    orientation="vertical"
)

Map_ENUI.add_layer_control()

Map_ENUI

In [ ]:
export_task = ee.batch.Export.image.toAsset(
    image=ENUI,
    description="ENUI_Tehran_2020",
    assetId="projects/bold-physics-478216-b4/assets/ENUI_Tehran_2020",
    region=roi,
    scale=30,
    maxPixels=1e13
)

export_task.start()

print("ENUI export task started.")